In [1]:
import torch
import torch.nn as nn

from floodvision.models.baseline_cnn import BaselineCNN

torch.manual_seed(42)

model = BaselineCNN(
    in_channels=2,
    out_channels=1,
    base_channels=16
)

print(model)

BaselineCNN(
  (encoder1): DoubleConv(
    (block): Sequential(
      (0): Conv2d(2, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (4): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
  )
  (encoder2): DoubleConv(
    (block): Sequential(
      (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (5): ReLU(inplac

In [2]:
dummy_input = torch.randn(
    1, 2, 512, 512
)

model.eval()

with torch.no_grad():

    logits = model(dummy_input)

print("Input shape:", dummy_input.shape)
print("Output shape:", logits.shape)

print("Minimum logit:", logits.min().item())
print("Maximum logit:", logits.max().item())

assert logits.shape == (1, 1, 512, 512)
assert torch.isfinite(logits).all()

print("Forward pass successful!")

Input shape: torch.Size([1, 2, 512, 512])
Output shape: torch.Size([1, 1, 512, 512])
Minimum logit: -0.10445480793714523
Maximum logit: -0.0924995020031929
Forward pass successful!


In [3]:
total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

model_size_mb = sum(
    p.numel() * p.element_size()
    for p in model.parameters()
) / (1024 ** 2)

print(f"Total parameters: {total_params:,}")

print(f"Trainable parameters: {trainable_params:,}")

print(f"Parameter size: {model_size_mb:.2f} MB")

Total parameters: 434,209
Trainable parameters: 434,209
Parameter size: 1.66 MB


In [4]:
from floodvision.models.blocks import DoubleConv


def shape_hook(name):

    def hook(module, inputs, output):

        print(
            f"{name:20s} "
            f"{str(tuple(inputs[0].shape)):25s}"
            f" -> {tuple(output.shape)}"
        )

    return hook


hooks = []

for name, module in model.named_children():

    if isinstance(
        module,
        (
            DoubleConv,
            nn.ConvTranspose2d,
            nn.MaxPool2d,
            nn.Conv2d
        )
    ):

        handle = module.register_forward_hook(
            shape_hook(name)
        )

        hooks.append(handle)


try:

    with torch.no_grad():
        output = model(dummy_input)

finally:

    for handle in hooks:
        handle.remove()

encoder1             (1, 2, 512, 512)          -> (1, 16, 512, 512)
pool                 (1, 16, 512, 512)         -> (1, 16, 256, 256)
encoder2             (1, 16, 256, 256)         -> (1, 32, 256, 256)
pool                 (1, 32, 256, 256)         -> (1, 32, 128, 128)
encoder3             (1, 32, 128, 128)         -> (1, 64, 128, 128)
pool                 (1, 64, 128, 128)         -> (1, 64, 64, 64)
bottleneck           (1, 64, 64, 64)           -> (1, 128, 64, 64)
up3                  (1, 128, 64, 64)          -> (1, 64, 128, 128)
decoder3             (1, 64, 128, 128)         -> (1, 64, 128, 128)
up2                  (1, 64, 128, 128)         -> (1, 32, 256, 256)
decoder2             (1, 32, 256, 256)         -> (1, 32, 256, 256)
up1                  (1, 32, 256, 256)         -> (1, 16, 512, 512)
decoder1             (1, 16, 512, 512)         -> (1, 16, 512, 512)
output               (1, 16, 512, 512)         -> (1, 1, 512, 512)


In [5]:
probabilities = torch.sigmoid(logits)

predictions = (
    probabilities >= 0.5
).float()

print("Probability shape:", probabilities.shape)

print("Minimum probability:", probabilities.min().item())
print("Maximum probability:", probabilities.max().item())

print("Prediction labels:", torch.unique(predictions))

Probability shape: torch.Size([1, 1, 512, 512])
Minimum probability: 0.47391000390052795
Maximum probability: 0.4768916070461273
Prediction labels: tensor([0.])
